# Athena sạch cho YouCook2: phần A (train lại trên trainsplit, chọn trên dev)

Theo `docs/UEMR_ATHENA_SACH_YC2.md` phần A. Code: nhánh `chien` của `UniAV-fixed` (repo này). So với lần train `athena.pth`
(`ov_E1`, commit `6552590`) **chỉ đổi dữ liệu**; mọi thứ khác là code cũ:

| | `ov_E1` (`athena.pth`, `2aef4406`) | lần này (`yc2clean_E1`) |
|---|---|---|
| train | 1.106 video `training` (gồm 104 dev) | 1.002 video trainsplit |
| chọn `best_seg` (R@0.5) / `best_cap` (CIDEr) | val 394 | dev 104 |
| mean câu IV2 (`caption_emb_iv2j*.npz`) | câu `training` cũ (gồm câu dev) + nhãn COIN train | câu trainsplit + nhãn COIN train |
| COIN test (chỉ in số, không chọn gì) | có | tắt (`coin.eval: false`) |

Kiến trúc, loss, siêu tham số, feature (`nguyenminh04/uniav-youcook2-data`), teacher Omni, seed `1234567891`, khởi tạo ngẫu nhiên,
`--set num_workers=4 train_cfg.ema_decay=0.999` (lệnh E1 trong `tools/colab_ov.sh`): giữ nguyên. **Val không được chấm ở đâu trong notebook này.**

**Ba file đã thêm/sửa trong repo** (commit "yc2-clean: train on trainsplit…"):
- `data/youcookii/annotations/youcookii_annotations_yc2clean.json`: annotation cũ, chỉ 104 video dev đổi `subset` thành `"dev"`.
- `configs/youcook2_event_yc2clean.yaml`: config cũ, đổi file annotation, `val_split: [dev]`, file câu sạch, tắt COIN test.
- `tools/embed_captions.py`: thêm 2 tham số cho `--joint` (file annotation, đuôi tên file ra) để tính mean câu chỉ trên trainsplit.

**Mỗi ô làm gì:** 1 đường dẫn → 2 lấy code, cài đặt → 3 tải feature và encoder → 4 kiểm danh sách video → 5 tính lại mean câu →
6 train → 7 export checkpoint + `SOURCE.json` → 8 in tóm tắt. Ô nào sai thì dừng ở `assert` với lời giải thích.

**Cách chạy:** Runtime **A100** (hoặc L4), **Run all**. Bị ngắt thì Run all lại: ô nào đã có đầu ra hợp lệ trên Drive thì bỏ qua
(câu: `captions_check.json`; train: `record.json` có `finished`; export: `SOURCE.json`). Train bị ngắt giữa chừng thì train lại từ đầu
(`train.py` không có resume). Thời gian: tải dữ liệu 10–20 phút, câu 2–3 phút, train 12 epoch khoảng 1–2 giờ.

**Đầu ra** (`MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/`): `split_check.json`, `train_videos.txt`, `captions_check.json`,
`caption_emb_iv2j_yc2clean.npz`, `run_yc2clean_E1/` (2 checkpoint, `record.json`, `train.log`), `<hash8>/athena_yc2clean.pth` + `SOURCE.json`.
Gửi lại log của **mọi ô** (nhất là ô 4, 5, 6, 7, 8).

In [1]:
# ===== 1. Drive, đường dẫn, hằng số =====
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, hashlib, shutil, subprocess, platform, time, tarfile

GIT_URL   = 'https://github.com/im-xiaoming/UniAV-fixed.git'      # = im-xiaoming/Athena-Retriever
BRANCH, BASE_COMMIT = 'chien', '6552590'   # 6552590: commit train athena.pth (ov_E1), để ghi diff
REPO      = '/content/UniAV-fixed'
CFG       = 'configs/youcook2_event_yc2clean.yaml'
RUN       = 'yc2clean_E1'
RUN_ARGS  = ['--set', 'num_workers=4', 'train_cfg.ema_decay=0.999']   # đúng lệnh E1 của tools/colab_ov.sh
N_EPOCHS  = 12                                                         # --epochs 10 (mặc định) + warmup 2, như ov_E1
ANNO      = 'data/youcookii/annotations/youcookii_annotations_yc2clean.json'
ANNO_SHA  = '0483e7bb07ffe522199eeec1af5b8b68819411820fce16f835a97396b3bf8215'
OLD_SHA8  = '2aef4406'
FUS       = '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr'
SUBSET    = f'{FUS}/omni_cache/common/subset_videos.json'
META      = '/content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII/metadata'
OUT       = f'{FUS}/athena_sach'        # mọi đầu ra phần A; không ghi vào chỗ của Athena cũ
OUT_RUN   = f'{OUT}/run_{RUN}'
os.makedirs(OUT, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN') or (open('/content/hf_token').read().strip() if os.path.exists('/content/hf_token') else None)
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None   # các repo HF dùng ở đây công khai

def sha256(p, n=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    assert r.returncode == 0, f'{cmd}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}'
    return r.stdout.strip()

assert os.path.isfile(SUBSET) and os.path.isdir(META), f'Drive thiếu {SUBSET} hoặc {META}'

Mounted at /content/drive


In [3]:
# ===== 2. Code nhánh chien (bản mới nhất trên GitHub), cài đặt, môi trường =====
if not os.path.isdir(f'{REPO}/.git'):
    sh(f'git clone -q -b {BRANCH} {GIT_URL} {REPO}')
sh(f'git fetch -q origin {BRANCH}', REPO)
sh(f'git checkout -q -B {BRANCH} origin/{BRANCH}', REPO)
HEAD = sh('git rev-parse --short HEAD', REPO)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sh(f'{sys.executable} -m pip install -q pycocoevalcap h5py tensorboard')
if subprocess.run([sys.executable, '-c', 'import torch, nms_1d_cpu'], capture_output=True).returncode:
    sh(f'{sys.executable} setup.py install --user > /content/nms_build.log 2>&1', f'{REPO}/libs/utils')
sh(f'{sys.executable} -c "import torch, nms_1d_cpu"')
import numpy as np, torch
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
java = subprocess.run('java -version', shell=True, capture_output=True, text=True).stderr.strip().splitlines()[:1]
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'numpy': np.__version__,
       'gpu': gpu or 'không có', 'java (METEOR)': java[0] if java else 'không có: METEOR bỏ qua', 'code': f'{BRANCH} @ {HEAD}',
       'tracked files changed': sh('git status --porcelain --untracked-files=no', REPO) or 'none'}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
assert os.path.exists(f'{REPO}/{CFG}') and os.path.exists(f'{REPO}/{ANNO}'), f'nhánh {BRANCH} chưa có {CFG}: đã push chưa?'
assert torch.cuda.is_available(), 'cần GPU'
assert int(np.__version__.split('.')[0]) >= 2, 'checkpoint Athena cần numpy 2.x'

{
 "python": "3.13.15",
 "torch": "2.11.0+cu130",
 "cuda": "13.0",
 "numpy": "2.1.3",
 "gpu": "NVIDIA A100-SXM4-40GB, 40960 MiB",
 "java (METEOR)": "openjdk version \"21.0.12.1\" 2026-08-18",
 "code": "chien @ 39d8161",
 "tracked files changed": "none"
}


In [4]:
# ===== 3. Dữ liệu: feature + teacher (HF dataset), encoder chữ IV2, athena.pth cũ để so định dạng =====
from huggingface_hub import snapshot_download, hf_hub_download
FEATS = f'{REPO}/data/youcookii/iv2_feats'
TEACHER = f'{REPO}/data/youcookii/omni_emb_full.npz'
if not (os.path.exists(f'{FEATS}/.done') and os.path.exists(TEACHER)):
    d = snapshot_download('nguyenminh04/uniav-youcook2-data', repo_type='dataset', local_dir='/content/hf_yc',
                          allow_patterns=['iv2_feats/*', 'teacher/*'], token=HF_TOKEN)
    os.makedirs(FEATS, exist_ok=True)
    for t in sorted(glob.glob(f'{d}/iv2_feats/*.tar')):
        with tarfile.open(t) as tf:
            tf.extractall(FEATS)
    shutil.copy(f'{d}/teacher/omni_emb_full.npz', TEACHER)
    open(f'{FEATS}/.done', 'w').write('ok')
IV2 = hf_hub_download('nguyenminh04/athena', 'internvideo2/InternVideo2-stage2_1b-224p-f4.pt', local_dir=f'{REPO}/ckpt', token=HF_TOKEN)
OLD = hf_hub_download('nguyenminh04/athena', 'athena.pth', local_dir='/content/old_ckpt', token=HF_TOKEN)
OLD_SHA = sha256(OLD)
n_feat = len(glob.glob(f'{FEATS}/*.npz'))
print('feature:', n_feat, '| teacher:', os.path.getsize(TEACHER) // 2**20, 'MB | IV2:', IV2, '| athena.pth cũ:', OLD_SHA[:16])
assert n_feat >= 1500, n_feat
assert IV2 == f'{REPO}/ckpt/internvideo2/InternVideo2-stage2_1b-224p-f4.pt'   # đường dẫn tools/embed_captions.py đọc
assert OLD_SHA.startswith(OLD_SHA8), f'athena.pth trên HF không phải {OLD_SHA8}'

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_2790/2385767249.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(FEATS)


internvideo2/InternVideo2-stage2_1b-224p(…): reconstructing file:   0%|          |  0.00B / 2.82GB            

internvideo2/InternVideo2-stage2_1b-224p(…): downloading bytes:           |  0.00B            

athena.pth: reconstructing file:   0%|          |  0.00B /  278MB            

athena.pth: downloading bytes:           |  0.00B            

feature: 1500 | teacher: 126 MB | IV2: /content/UniAV-fixed/ckpt/internvideo2/InternVideo2-stage2_1b-224p-f4.pt | athena.pth cũ: 2aef4406818ec0e3


In [5]:
# ===== 4. Kiểm split (mục A.1): đúng danh sách của subset_videos.json, train = 1.002 trainsplit, không dev, không val =====
import yaml
from libs.datasets import YouCook2CaptionDataset
assert sha256(ANNO) == ANNO_SHA, 'file annotation yc2clean bị đổi'
anno = json.load(open(ANNO))['database']
cfg = yaml.safe_load(open(CFG))
assert cfg['train_split'] == ['training'] and cfg['val_split'] == ['dev'], (cfg['train_split'], cfg['val_split'])

sub = json.load(open(SUBSET))['videos']
S_DEV = {r['video_id'] for r in sub if r['split'] == 'dev'}
S_VAL = {r['video_id'] for r in sub if r['split'] == 'val'}
vids = lambda p: {os.path.splitext(os.path.basename(json.loads(l)['video']))[0] for l in open(p) if l.strip()}
M_TR, M_DEV, M_VAL = (vids(f'{META}/{f}') for f in ('train_omni_video.trainsplit.jsonl', 'train_omni_video.dev.jsonl',
                                                     'val_omni_video.jsonl'))
# đúng các video train.py sẽ đọc (file câu sạch chưa có ở bước này; khoá câu giống hệt file cũ)
dcfg = dict(cfg['dataset'], caption_emb_file='./data/youcookii/caption_emb_iv2j.npz', omni_emb_file=None)
ds_tr = YouCook2CaptionDataset(True, cfg['train_split'], **dcfg)
ds_dv = YouCook2CaptionDataset(False, cfg['val_split'], **dcfg)
TR = {x['id'] for x in ds_tr.data_list}
DV = {x['id'] for x in ds_dv.data_list}
cap_tr, cap_dv = sum(x['n_cap'] for x in ds_tr.data_list), sum(x['n_cap'] for x in ds_dv.data_list)
VAL_ALL = {v for v, x in anno.items() if x['subset'] == 'validation'}
SPLIT = {'train videos': len(TR), 'train captions (<=16/video)': cap_tr, 'dev videos': len(DV), 'dev captions': cap_dv,
         'unique train captions (pool)': len(ds_tr.pool_text),
         'train == manifest trainsplit': TR == M_TR, 'dev == subset_videos dev': DV == S_DEV, 'dev == manifest dev': DV == M_DEV,
         'train ∩ (dev ∪ val)': len(TR & (S_DEV | S_VAL | M_VAL | VAL_ALL)), 'dev ∩ val': len(DV & (S_VAL | VAL_ALL)),
         'subset_videos: dev / val': (len(S_DEV), len(S_VAL))}
if os.path.exists(f'{META}/dev_videos.txt'):
    SPLIT['dev == dev_videos.txt'] = DV == {l.strip() for l in open(f'{META}/dev_videos.txt') if l.strip()}
print(json.dumps(SPLIT, indent=1, ensure_ascii=False))
json.dump(SPLIT, open(f'{OUT}/split_check.json', 'w'), indent=1, ensure_ascii=False)
open(f'{OUT}/train_videos.txt', 'w').write('\n'.join(sorted(TR)) + '\n')
assert len(TR) == 1002 and TR == M_TR, 'train phải đúng 1.002 video trainsplit'
assert len(DV) == 104 and DV == S_DEV == M_DEV, 'dev phải đúng 104 video dev của subset_videos.json'
assert SPLIT['train ∩ (dev ∪ val)'] == 0 and SPLIT['dev ∩ val'] == 0
assert SPLIT.get('dev == dev_videos.txt', True)

{
 "train videos": 1002,
 "train captions (<=16/video)": 7790,
 "dev videos": 104,
 "dev captions": 773,
 "unique train captions (pool)": 7512,
 "train == manifest trainsplit": true,
 "dev == subset_videos dev": true,
 "dev == manifest dev": true,
 "train ∩ (dev ∪ val)": 0,
 "dev ∩ val": 0,
 "subset_videos: dev / val": [
  104,
  394
 ],
 "dev == dev_videos.txt": true
}


In [6]:
  # ===== 5. Mean câu IV2 (mục A.3): tính lại chỉ trên câu trainsplit =====
# Trước đó: chạy lại đúng lệnh cũ (annotation trainval) và so với file đang có trong git, để chắc encode tái lập được.
CAP_OLD, CAP_REP, CAP_NEW = (f'data/youcookii/caption_emb_iv2j{t}.npz' for t in ('', '_repro', '_yc2clean'))
DRV_CAP, CHK = f'{OUT}/caption_emb_iv2j_yc2clean.npz', f'{OUT}/captions_check.json'
if os.path.exists(CHK) and os.path.exists(DRV_CAP) and sha256(DRV_CAP) == json.load(open(CHK))['sha256']:
    shutil.copy(DRV_CAP, CAP_NEW)
    CAPS = json.load(open(CHK))
else:
    sh(f'{sys.executable} tools/embed_captions.py --joint data/youcookii/annotations/youcookii_annotations_trainval.json _repro', REPO)
    sh(f'{sys.executable} tools/embed_captions.py --joint {ANNO} _yc2clean', REPO)
    o, r, n = (np.load(p, allow_pickle=True) for p in (CAP_OLD, CAP_REP, CAP_NEW))
    f32 = lambda a: a.astype(np.float32)
    old_mean = torch.load(OLD, map_location='cpu', weights_only=False)['caption_pool']['mean']
    have = {os.path.basename(f)[:-4] for f in glob.glob(f'{FEATS}/*.npz')}
    uniq = lambda db, s: len({a['sentence'] for v, x in db.items() if x['subset'] == s and v in have for a in x['annotations']})
    db_old = json.load(open('data/youcookii/annotations/youcookii_annotations_trainval.json'))['database']
    CAPS = {'keys equal (old / repro / clean)': bool(list(o['keys']) == list(r['keys']) == list(n['keys'])),
            'repro: min cos emb vs git file': float((f32(o['emb']) * f32(r['emb'])).sum(1).min()),
            'repro: max |mean - git mean|': float(np.abs(o['mean'] - r['mean']).max()),
            'git mean == athena.pth caption_pool.mean': bool(np.allclose(o['mean'], old_mean, atol=1e-6)),
            'unique captions in mean: old (training cũ) / clean (trainsplit)': (uniq(db_old, 'training'), uniq(anno, 'training')),
            '|mean_clean - mean_old|': float(np.linalg.norm(n['mean'] - o['mean'])),
            'clean: min cos emb vs old': float((f32(o['emb']) * f32(n['emb'])).sum(1).min()),
            'sha256': sha256(CAP_NEW)}
    shutil.copy(CAP_NEW, DRV_CAP)
    json.dump(CAPS, open(CHK, 'w'), indent=1, ensure_ascii=False)
print(json.dumps(CAPS, indent=1, ensure_ascii=False))
assert CAPS['keys equal (old / repro / clean)'] and CAPS['git mean == athena.pth caption_pool.mean']
assert CAPS['repro: min cos emb vs git file'] > 0.999 and CAPS['repro: max |mean - git mean|'] < 1e-3, 'encode không tái lập được file cũ: dừng'
assert CAPS['|mean_clean - mean_old|'] > 0 and sha256(CAP_NEW) == CAPS['sha256']

{
 "keys equal (old / repro / clean)": true,
 "repro: min cos emb vs git file": 0.999886691570282,
 "repro: max |mean - git mean|": 2.9802322387695312e-08,
 "git mean == athena.pth caption_pool.mean": true,
 "unique captions in mean: old (training cũ) / clean (trainsplit)": [
  8218,
  7512
 ],
 "|mean_clean - mean_old|": 0.0008060159161686897,
 "clean: min cos emb vs old": 0.999856173992157,
 "sha256": "d7b067d7ff4a6ba2d550235410d8d78db6d50c2cab30fa51cfbd1291bd5cbd5c"
}


In [7]:
  # ===== 6. Train yc2clean_E1 (chọn best_seg theo R@0.5 dev, best_cap theo CIDEr dev) =====
RUN_DIR = f'{REPO}/ckpt/{RUN}'
CKS = ('best_seg.pth.tar', 'best_cap.pth.tar')
def finished():
    p = f'{OUT_RUN}/record.json'
    return os.path.exists(p) and json.load(open(p)).get('finished') and all(os.path.exists(f'{OUT_RUN}/{c}') for c in CKS)
if finished():
    os.makedirs(RUN_DIR, exist_ok=True)
    for c in CKS:
        shutil.copy(f'{OUT_RUN}/{c}', f'{RUN_DIR}/{c}')
    os.makedirs(f'{REPO}/experiments/runs', exist_ok=True)
    shutil.copy(f'{OUT_RUN}/record.json', f'{REPO}/experiments/runs/colab-{RUN}.json')
    print('đã train xong trước đó: đọc lại từ', OUT_RUN)
else:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    for p in glob.glob(f'{REPO}/experiments/runs/*-{RUN}.json'):
        os.remove(p)
    os.makedirs(OUT_RUN, exist_ok=True)
    cmd = [sys.executable, '-u', 'train.py', CFG, '--output', RUN, '--note', 'UEMR_ATHENA_SACH_YC2 phan A'] + RUN_ARGS
    print(' '.join(cmd), flush=True)
    t0 = time.time()
    with open('/content/train.log', 'w') as lf:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
    assert p.wait() == 0, 'train.py lỗi: xem log ở trên'
    print('train: %.0f phút' % ((time.time() - t0) / 60))
    shutil.copy('/content/train.log', f'{OUT_RUN}/train.log')
    for c in CKS:
        shutil.copy(f'{RUN_DIR}/{c}', f'{OUT_RUN}/{c}')
    shutil.copy(glob.glob(f'{REPO}/experiments/runs/*-{RUN}.json')[0], f'{OUT_RUN}/record.json')

rec = json.load(open(f'{OUT_RUN}/record.json'))
log = open(f'{OUT_RUN}/train.log').read()
c = rec['config']
print('best:', json.dumps(rec['best']), '| git:', rec['git'], '| overrides:', rec['overrides'])
assert 'Data         : 1002 train / 104 val videos' in log, 'log không in 1002 train / 104 dev'
assert c['train_split'] == ['training'] and c['val_split'] == ['dev'] and c['dataset']['json_file'].endswith('yc2clean.json')
assert c['dataset']['caption_emb_file'].endswith('_yc2clean.npz') and not c['coin']['train'] and not c['coin']['eval']
assert c['train_cfg']['ema_decay'] == 0.999 and c['init_rand_seed'] == 1234567891 and c['num_workers'] == 4
assert 'validation' not in json.dumps(c), 'config còn nhắc tới validation'
TRAIN_COMMIT = rec['git']['commit']   # commit code lúc train (= HEAD nếu vừa train ở phiên này)
assert rec['git']['dirty'] is False, rec['git']
assert rec['finished'] and len(rec['history']) == N_EPOCHS and rec['final_eval'], 'run chưa xong'

/usr/bin/python3 -u train.py configs/youcook2_event_yc2clean.yaml --output yc2clean_E1 --note UEMR_ATHENA_SACH_YC2 phan A --set num_workers=4 train_cfg.ema_decay=0.999
Override     : num_workers = 4
Override     : train_cfg.ema_decay = 0.999
Omni teacher : 7483/7512 caption vectors, 7787 clip vectors from ./data/youcookii/omni_emb_full.npz
Data         : 1002 train / 104 val videos
Features     : InternVideo2 v768 + BEATs a768, 1 row(s)/s, visual 768 + audio 768 channels
Parameters   : 70.5M
Caption pool : 7512 unique train captions
Output       : ./ckpt/yc2clean_E1
Run record   : /content/UniAV-fixed/experiments/runs/59dc73b4ee4e-yc2clean_E1.json (commit 39d8161)
EMA         : decay 0.999, evaluated and saved instead of the raw weights
Legend       : S = new best R@0.5, C = new best CIDEr (each saves its checkpoint)
               cos_rand = cosine to a random caption (floor), txt_sim = similarity of
               chosen caption to GT caption (InternVideo2 text space), CIDEr = word o

In [8]:
# ===== 7. Export đúng định dạng athena.pth, so với bản cũ, sha256, SOURCE.json (mục A.5) =====
API = f'{REPO}/ckpt/api/athena_yc2clean.pth'
if not os.path.exists(API):
    sh(f'{sys.executable} tools/export_api_ckpt.py {RUN} --out {API}', REPO)
    ck = torch.load(API, map_location='cpu', weights_only=False)
    if 'state_dict_seg' not in ck:   # best_seg cùng epoch với best_cap: export chỉ lưu một bộ; uniav_app cần cả hai khoá
        ck['state_dict_seg'], ck['epoch_seg'] = ck['state_dict'], ck['epoch']
        torch.save(ck, API)
ck = torch.load(API, map_location='cpu', weights_only=False)
old = torch.load(OLD, map_location='cpu', weights_only=False)
shapes = lambda sd: {k: tuple(v.shape) for k, v in sd.items()}
FMT = {'top keys equal': set(ck) == set(old), 'missing top keys': sorted(set(old) - set(ck)),
       'state_dict shapes equal': shapes(ck['state_dict']) == shapes(old['state_dict']),
       'state_dict_seg shapes equal': shapes(ck['state_dict_seg']) == shapes(old['state_dict_seg']),
       'config equal (dataset + model)': ck['config'] == old['config'],
       'caption_pool size new / old': (len(ck['caption_pool']['emb']), len(old['caption_pool']['emb'])),
       'caption_pool file': ck['caption_pool']['file'],
       'caption_pool mean changed': not np.allclose(ck['caption_pool']['mean'], old['caption_pool']['mean']),
       'run / commit': (ck['run'], ck['commit']), 'epoch_seg / epoch': (ck['epoch_seg'], ck['epoch'])}
print(json.dumps(FMT, indent=1, ensure_ascii=False, default=str))
assert FMT['top keys equal'] and FMT['state_dict shapes equal'] and FMT['state_dict_seg shapes equal'] and FMT['config equal (dataset + model)']
assert FMT['caption_pool size new / old'][0] == FMT['caption_pool size new / old'][1] and FMT['caption_pool file'] == 'caption_emb_iv2j_yc2clean.npz'
assert FMT['caption_pool mean changed'] and ck['run'] == RUN and ck['commit'] == TRAIN_COMMIT

SHA = sha256(API); H8 = SHA[:8]
DST = f'{OUT}/{H8}'
os.makedirs(DST, exist_ok=True)
if not (os.path.exists(f'{DST}/athena_yc2clean.pth') and sha256(f'{DST}/athena_yc2clean.pth') == SHA):
    shutil.copy(API, f'{DST}/athena_yc2clean.pth')
hist = {h['epoch']: h for h in rec['history']}
KEEP = ('R@0.3', 'R@0.5', 'R@0.7', 'mIoU', 'seg/vid', 'CIDEr', 'txt_sim', 'G R1@0.5', 'G R1@0.7', 'G mIoU')
dev_row = lambda e: {k: round(hist[e][k], 3) for k in KEEP if k in hist[e]}
e_seg, e_cap = rec['best']['R@0.5']['epoch'], rec['best']['CIDEr']['epoch']
assert (e_seg, e_cap) == (ck['epoch_seg'], ck['epoch']), ((e_seg, e_cap), (ck['epoch_seg'], ck['epoch']))
excl = "':(exclude)data/youcookii/annotations/youcookii_annotations_yc2clean.json'"
open(f'{DST}/diff_vs_{BASE_COMMIT}.patch', 'w').write(sh(f'git diff {BASE_COMMIT} {TRAIN_COMMIT} -- configs libs tools train_event.py train.py {excl}', REPO) + '\n')
shutil.copy(f'{REPO}/{CFG}', f'{DST}/{os.path.basename(CFG)}')
for f in ('train_videos.txt', 'split_check.json', 'captions_check.json'):
    shutil.copy(f'{OUT}/{f}', f'{DST}/{f}')
SOURCE = {
    'checkpoint': f'{DST}/athena_yc2clean.pth', 'sha256': SHA, 'hash8': H8, 'run': RUN, 'replaces': f'athena.pth ({OLD_SHA[:16]}, ov_E1)',
    'code': {'repo': GIT_URL, 'branch': BRANCH, 'commit': rec['git']['commit'], 'dirty': rec['git']['dirty'],
             'base_commit (ov_E1)': BASE_COMMIT, 'diff_stat': sh(f'git diff --stat {BASE_COMMIT} {TRAIN_COMMIT} -- configs libs tools data train_event.py train.py', REPO),
             'diff': f'diff_vs_{BASE_COMMIT}.patch (annotation file excluded: it differs only in subset of 104 dev videos)',
             'config': CFG, 'argv': rec['argv']},
    'data': {'annotation': ANNO, 'annotation_sha256': ANNO_SHA, 'train_videos': len(TR), 'train_captions': cap_tr,
             'dev_videos': len(DV), 'dev_captions': cap_dv, 'train_video_list': 'train_videos.txt',
             'validation': 'không dùng: không train, không chấm, không chọn epoch / checkpoint, không tính thống kê',
             'caption_emb': {'file': 'caption_emb_iv2j_yc2clean.npz', 'sha256': CAPS['sha256'],
                             'mean': '0.5 * mean(unique trainsplit captions) + 0.5 * mean(COIN train labels), như embed_captions.py --joint'},
             'features': 'nguyenminh04/uniav-youcook2-data iv2_feats (v768 + a768), teacher/omni_emb_full.npz (chỉ video train)'},
    'init': 'ngẫu nhiên, init_rand_seed 1234567891, như ov_E1; không nạp model_epoch_006.pth.tar',
    'selection': {
        'state_dict_seg': {'by': 'R@0.5 cao nhất trên dev (best_seg)', 'epoch': e_seg, 'dev': dev_row(e_seg)},
        'state_dict': {'by': 'CIDEr cao nhất trên dev (best_cap)', 'epoch': e_cap, 'dev': dev_row(e_cap),
                       'final_eval_dev': {k: round(v, 3) for k, v in rec['final_eval'].items() if isinstance(v, (int, float))}}},
    'history_dev': [dict(epoch=e, **dev_row(e)) for e in sorted(hist)],
    'format_check': FMT,
    'notes': ['Caption bank của API vẫn là 8218 câu train cũ (athena/assets/caption_pool.npz, cần cho định dạng uniav_app): '
              'gồm text câu dev, vector căn theo mean sạch. Chỉ dùng khi in caption; UEMR không dùng.',
              'COIN test tắt (coin.eval false): chỉ là số in thêm, không ảnh hưởng train hay chọn checkpoint.'],
    'env': ENV, 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
json.dump(SOURCE, open(f'{DST}/SOURCE.json', 'w'), indent=1, ensure_ascii=False, default=str)
assert sha256(f'{DST}/athena_yc2clean.pth') == SHA

{
 "top keys equal": true,
 "missing top keys": [],
 "state_dict shapes equal": true,
 "state_dict_seg shapes equal": true,
 "config equal (dataset + model)": true,
 "caption_pool size new / old": [
  8218,
  8218
 ],
 "caption_pool file": "caption_emb_iv2j_yc2clean.npz",
 "caption_pool mean changed": true,
 "run / commit": [
  "yc2clean_E1",
  "39d8161"
 ],
 "epoch_seg / epoch": [
  8,
  10
 ]
}


In [9]:
# ===== 8. Tóm tắt để ghi nhật ký (mục C) =====
print('checkpoint :', SOURCE['checkpoint'])
print('sha256     :', SHA, '| hash8', H8)
print('code       :', BRANCH, TRAIN_COMMIT, '| base', BASE_COMMIT)
print('data       : train %d video / %d câu, dev %d video / %d câu, val không dùng' % (len(TR), cap_tr, len(DV), cap_dv))
for k, v in SOURCE['selection'].items():
    print('%-15s: epoch %d (%s) | dev %s' % (k, v['epoch'], v['by'], json.dumps(v['dev'])))
print('dev theo epoch:')
for h in SOURCE['history_dev']:
    print('  ', json.dumps(h))
assert os.path.exists(f'{DST}/SOURCE.json')

checkpoint : /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/athena_yc2clean.pth
sha256     : 961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58 | hash8 961f1a89
code       : chien 39d8161 | base 6552590
data       : train 1002 video / 7790 câu, dev 104 video / 773 câu, val không dùng
state_dict_seg : epoch 8 (R@0.5 cao nhất trên dev (best_seg)) | dev {"R@0.3": 75.809, "R@0.5": 56.145, "R@0.7": 32.083, "mIoU": 52.105, "seg/vid": 7.433, "CIDEr": 87.586, "txt_sim": 0.52, "G R1@0.5": 47.089, "G R1@0.7": 31.177, "G mIoU": 43.708}
state_dict     : epoch 10 (CIDEr cao nhất trên dev (best_cap)) | dev {"R@0.3": 73.868, "R@0.5": 54.592, "R@0.7": 30.789, "mIoU": 50.711, "seg/vid": 7.433, "CIDEr": 89.898, "txt_sim": 0.528, "G R1@0.5": 45.796, "G R1@0.7": 29.107, "G mIoU": 42.594}
dev theo epoch:
   {"epoch": 1, "R@0.3": 68.564, "R@0.5": 45.666, "R@0.7": 21.475, "mIoU": 45.266, "seg/vid": 7.433, "CIDEr": 56.402, "txt_sim": 0.505, "G R1@0.5": 29.237, "